In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

To make sure the new installed libraries are used, **sometimes it's required to restart the notebook runtime**. The next cell will force the **runtime to crash, so you'll need to connect again and run the code starting from here**. Thanks to this trick, **we will be able to run our virtual screen.**

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [ ]:
!pip install gymnasium[box2d]
!pip install stable-baselines3[extra]

In [ ]:
!pip install tensorflow

In [ ]:
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

In [ ]:
import gymnasium as gym

# First, we create our environment called LunarLander-v2
env = gym.make("CartPole-v1")

# Then we reset this environment
observation, info = env.reset()

for _ in range(20):
  # Take a random action
  action = env.action_space.sample()
  print("Action taken:", action)

  # Do this action in the environment and get
  # next_state, reward, terminated, truncated and info
  observation, reward, terminated, truncated, info = env.step(action)

  # If the game is terminated (in our case we land, crashed) or truncated (timeout)
  if terminated or truncated:
      # Reset the environment
      print("Environment is reset")
      observation, info = env.reset()

env.close()

In [ ]:
# We create our environment with gym.make("<name_of_the_environment>")
env = gym.make("CartPole-v1")
env.reset()
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample()) # Get a random observation

In [ ]:
# Create the environment
env = make_vec_env("CartPole-v1", n_envs=16)

In [ ]:

model = DQN(
    policy = 'MlpPolicy',
    env = env,
    learning_rate = 0.0001,
    buffer_size = 1000000,
    learning_starts = 100,
    batch_size = 20,
    gamma = 0.999,
    train_freq=(1,"step"),
    device='cuda',
    verbose=0)

In [ ]:
#Adding Tensboard Logs for testing
model = DQN(
    policy = 'MlpPolicy',
    env = env,
    learning_rate = 0.0001,
    buffer_size = 1000000,
    learning_starts = 100,
    batch_size = 20,
    gamma = 0.999,
    train_freq=(1,"step"),
    device='cuda',
    verbose=0,
    tensorboard_log="./tensorboard_logs/")
#DQN policy appears to need quite a bit more adjusting to actually make it good vs PPO.
#Model reward increased from 9.5 to 157 when training frequency was changed from every 5 steps to every 1 step (v1)
#Model reward increased from 157 tot 171 by decreasing batch size from 32 to 20. #retrain 2 (except this was overwritten)
#Using same number of timesteps, rewards dropped from 171 to 9.5 when learning rate was changed from 0.0001 to 0.00001
#Movig to 3M steps and back to 0.0001 learn rate did not increase rewards as expected.  This is wierd.
#Moving back to 1M steps didn't fix anything
#Back to 0.0001 learning rate with batch size 20 and 1m timesteps.  Maybe I forgot rerun the model definition section
#Reward with this setting is highly variable, dropped to 92
#Dropped to 10.10 when timesteps changed to 3m
#linear Loss Growth noticed for first 1M time steps, this would explain why training over 3M time steps gets worse.  Model can be improved significantly by cutting training off after 500K timesteps
#I do not know why this is, I would need to investigate to give a compelling reason for the homework submission.
#HW says to modify the agent itself, and I have not done much of this using Stable Baselines3 yet, I will need to get into this.  I have a feeling I have maximized reward/loss using training parameters

#Note default is [64,64] eg 2 hidden layers of neurons at 64 neurons each
#Double the size, then add a 3rd hidden layer, call your optimization done for this problem after that is complete


In [ ]:
policy_kwargs = dict(net_arch=[128,128])
model = DQN(
    policy = 'MlpPolicy',
    policy_kwargs=policy_kwargs,
    env = env,
    learning_rate = 0.0001,
    buffer_size = 1000000,
    learning_starts = 100,
    batch_size = 20,
    gamma = 0.999,
    train_freq=(1,"step"),
    device='cuda',
    verbose=0,
    tensorboard_log="./tensorboard_logs/")

In [ ]:
policy_kwargs = dict(net_arch=[128,128,128])
model = DQN(
    policy = 'MlpPolicy',
    policy_kwargs=policy_kwargs,
    env = env,
    learning_rate = 0.0001,
    buffer_size = 1000000,
    learning_starts = 100,
    batch_size = 20,
    gamma = 0.999,
    train_freq=(1,"step"),
    device='cuda',
    verbose=0,
    tensorboard_log="./tensorboard_logs/")

In [ ]:
model.learn(total_timesteps=1000000)
# Save the model
model_name = "CartPole-v1"
model.save(model_name)

In [ ]:
model.learn(total_timesteps=1000000)
# Save the model
model_name = "CartPole-v1_retrain_wider_net"
model.save(model_name)

In [ ]:
model.learn(total_timesteps=1000000)
# Save the model
model_name = "CartPole-v1_retrain_wider_deeper_net"
model.save(model_name)

In [ ]:
model.learn(total_timesteps=3000000)
# Save the model
model_name = "CartPole-v1_retrain_3x_timesteps"
model.save(model_name)

In [ ]:
eval_env = Monitor(gym.make("CartPole-v1", render_mode='rgb_array'))
env.render()  # This will render the environment on the screen
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

#Need to add loss plot

In [ ]:
from tensorboard.backend.event_processing import event_accumulator
import matplotlib.pyplot as plt
event_file="/content/tensorboard_logs/DQN_3/events.out.tfevents.1745878990.d13e86fd0d43.1928.2"
ea= event_reader= event_accumulator.EventAccumulator(event_file)
ea.Reload()
print(ea.Tags())
losses=ea.Scalars('train/loss')
#print(losses)
rewards=ea.Scalars('rollout/ep_rew_mean')
reward_steps= [e.step for e in rewards]
rewards_values=[e.value for e in rewards]

loss_steps=[e.step for e in losses]
losses=[e.value for e in losses]

plt.figure(figsize=(10, 5))
plt.plot(loss_steps, losses)
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('Training Loss Vs Steps')
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(reward_steps, rewards_values)
plt.xlabel('Reward_Steps')
plt.ylabel('Values')
plt.title('Reward_Steps Vs Values')
plt.show()